In [1]:
# ============================================================
# 11_fairness_metrics.ipynb
# Additional Fairness Metrics
#
# Metrics:
#   1. Demographic Parity Difference (DPD)
#   2. Equal Opportunity Difference (EOD)
#   3. Equalized Odds (EO)
#   4. Predictive Parity
# Stratified by: sex × age_group
# ============================================================


# ─────────────────────────────────────────────
# Cell 1 | Imports
# ─────────────────────────────────────────────
import os
import warnings
import numpy as np
import pandas as pd
import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

warnings.filterwarnings('ignore')

BASE_OUT = os.path.join('..', 'outputs')
BASE_MOD = os.path.join('..', 'outputs', 'models')
BASE_FIG = os.path.join('..', 'outputs', 'figures')

DISEASE_REGISTRY = {
    'htn': {
        'target'      : 'HE_HP',
        'label'       : 'Hypertension',
        'feature_cols': ['BD1_11', 'BD2_1', 'BS3_1',
                         'BE3_75', 'BE3_91', 'pa_aerobic',
                         'L_BR_FQ', 'BH1',
                         'HE_BMI', 'HE_wc', 'HE_wt',
                         'N_NA', 'N_K'],
    },
    'dm': {
        'target'      : 'HE_DM_HbA1c',
        'label'       : 'Diabetes',
        'feature_cols': ['BD1_11', 'BD2_1', 'BS3_1',
                         'BE3_75', 'BE3_91', 'pa_aerobic',
                         'L_BR_FQ', 'BH1',
                         'HE_BMI', 'HE_wc', 'HE_wt',
                         'N_SUGAR', 'N_CHO', 'N_EN'],
    },
    'dys': {
        'target'      : 'HE_HCHOL',
        'label'       : 'Dyslipidemia',
        'feature_cols': ['BD1_11', 'BD2_1', 'BS3_1',
                         'BE3_75', 'BE3_91', 'pa_aerobic',
                         'L_BR_FQ', 'BH1',
                         'HE_BMI', 'HE_wc', 'HE_wt',
                         'N_FAT', 'N_CHOL'],
    },
}

NON_ACTIONABLE = [
    'ID', 'year', 'age_group', 'age', 'sex',
    'edu', 'incm', 'ho_incm',
    'BE3_71', 'BE3_81', 'BP1', 'mh_stress',
    'HE_obe', 'BO1_1', 'BO1_2', 'BO1_3',
]

RANDOM_SEED = 42
AGE_BINS    = [18, 44, 64, 120]
AGE_LABELS  = ['young', 'middle', 'elderly']

print("Configuration loaded.")


# ─────────────────────────────────────────────
# Cell 2 | Fairness Metric Functions
# ─────────────────────────────────────────────
def demographic_parity_diff(y_pred, group):
    """
    DPD = P(Ŷ=1|A=a) - P(Ŷ=1|A=b)
    Max pairwise difference across groups.
    0 = perfect parity.
    """
    groups  = np.unique(group)
    rates   = {g: y_pred[group == g].mean()
               for g in groups}
    max_dpd = max(
        abs(rates[g1] - rates[g2])
        for i, g1 in enumerate(groups)
        for g2 in groups[i+1:]
    )
    return round(max_dpd, 4), rates


def equal_opportunity_diff(y_true, y_pred, group):
    """
    EOD = TPR_a - TPR_b  (among positives)
    Max pairwise TPR difference.
    """
    groups = np.unique(group)
    tprs   = {}
    for g in groups:
        mask   = group == g
        y_t    = y_true[mask]
        y_p    = y_pred[mask]
        pos    = y_t == 1
        if pos.sum() == 0:
            tprs[g] = np.nan
        else:
            tprs[g] = y_p[pos].mean()

    valid = {g: v for g, v in tprs.items()
             if not np.isnan(v)}
    if len(valid) < 2:
        return np.nan, tprs

    max_eod = max(
        abs(v1 - v2)
        for i, (g1, v1) in enumerate(valid.items())
        for g2, v2 in list(valid.items())[i+1:]
    )
    return round(max_eod, 4), tprs


def predictive_parity(y_true, y_pred, group):
    """
    PP = P(Y=1|Ŷ=1, A=a) - P(Y=1|Ŷ=1, A=b)
    Precision difference across groups.
    """
    groups = np.unique(group)
    ppv    = {}
    for g in groups:
        mask = group == g
        y_t  = y_true[mask]
        y_p  = y_pred[mask]
        pred_pos = y_p == 1
        if pred_pos.sum() == 0:
            ppv[g] = np.nan
        else:
            ppv[g] = y_t[pred_pos].mean()

    valid = {g: v for g, v in ppv.items()
             if not np.isnan(v)}
    if len(valid) < 2:
        return np.nan, ppv

    max_pp = max(
        abs(v1 - v2)
        for i, (g1, v1) in enumerate(valid.items())
        for g2, v2 in list(valid.items())[i+1:]
    )
    return round(max_pp, 4), ppv


# ─────────────────────────────────────────────
# Cell 3 | Main Fairness Loop
# ─────────────────────────────────────────────
fairness_rows = []

for d_key, d_info in DISEASE_REGISTRY.items():
    target = d_info['target']
    label  = d_info['label']
    feat   = d_info['feature_cols']

    df = pd.read_csv(
        os.path.join(BASE_OUT, f"{d_key}_total.csv")
    )
    df['age_group'] = pd.cut(
        df['age'], bins=AGE_BINS,
        labels=AGE_LABELS, right=True
    )

    drop  = NON_ACTIONABLE + [target]
    fcols = [c for c in df.columns
              if c not in drop and c in feat]
    X = df[fcols].astype(float)
    y = df[target].astype(int)

    model  = joblib.load(
        os.path.join(BASE_MOD,
                     f"xgb_{d_key}_total.joblib")
    )
    y_pred = model.predict(X)
    y_prob = model.predict_proba(X)[:, 1]

    print(f"\n{'='*58}")
    print(f"  {label}")
    print(f"{'='*58}")

    # Sensitive attributes to test
    for attr_name, attr_vals in [
        ('sex',       df['sex'].map(
            {1: 'Male', 2: 'Female'}).values),
        ('age_group', df['age_group'].astype(str).values),
    ]:
        dpd, dpd_rates = demographic_parity_diff(
            y_pred, attr_vals)
        eod, eod_tprs  = equal_opportunity_diff(
            y.values, y_pred, attr_vals)
        pp,  pp_ppv    = predictive_parity(
            y.values, y_pred, attr_vals)

        print(f"\n  Sensitive attribute: {attr_name}")
        print(f"    DPD (Demographic Parity Diff) : "
              f"{dpd:.4f}")
        print(f"    EOD (Equal Opportunity Diff)  : "
              f"{eod:.4f}")
        print(f"    PP  (Predictive Parity Diff)  : "
              f"{pp:.4f}")
        print(f"    Positive rates by group:")
        for g, r in dpd_rates.items():
            print(f"      {g}: {r:.4f}")

        fairness_rows.append({
            'Disease'  : label,
            'Attribute': attr_name,
            'DPD'      : dpd,
            'EOD'      : eod,
            'PP'       : pp,
        })

df_fair = pd.DataFrame(fairness_rows)

print("\n" + "="*65)
print("  FAIRNESS METRICS SUMMARY")
print("="*65)
print(df_fair.to_string(index=False))


# ─────────────────────────────────────────────
# Cell 4 | Fairness Visualization
# ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

metrics   = ['DPD', 'EOD', 'PP']
m_labels  = ['Demographic\nParity Diff',
              'Equal Opportunity\nDiff',
              'Predictive\nParity Diff']
colors    = ['#2c7bb6', '#1a9641', '#d7191c']
x         = np.arange(len(metrics))
width     = 0.25

for ax_i, attr in enumerate(['sex', 'age_group']):
    ax  = axes[ax_i]
    sub = df_fair[df_fair['Attribute'] == attr]

    for i, (d_key, d_info) in enumerate(
            DISEASE_REGISTRY.items()):
        vals = sub[sub['Disease'] == d_info['label']]\
               [metrics].values.flatten()
        bars = ax.bar(
            x + (i - 1) * width, vals,
            width, label=d_info['label'],
            color=colors[i], alpha=0.85,
            edgecolor='white'
        )
        for bar, val in zip(bars, vals):
            ax.text(
                bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.003,
                f"{val:.3f}",
                ha='center', va='bottom',
                fontsize=7
            )

    ax.axhline(0.1, color='red', linestyle=':',
               linewidth=1.2,
               label='Threshold (0.10)')
    ax.set_xticks(x)
    ax.set_xticklabels(m_labels, fontsize=10)
    ax.set_ylabel('Fairness Metric Value', fontsize=11)
    ax.set_ylim(0, max(
        df_fair[df_fair['Attribute'] == attr]
        [metrics].max().max() * 1.3, 0.15
    ))
    ax.set_title(
        f"Sensitive Attribute: "
        f"{'Sex' if attr == 'sex' else 'Age Group'}",
        fontsize=12, fontweight='bold'
    )
    ax.legend(fontsize=9)
    ax.grid(axis='y', linestyle='--', alpha=0.35)
    ax.spines[['top', 'right']].set_visible(False)

fig.suptitle(
    'Fairness Metrics by Sensitive Attribute\n'
    '(DPD=0 / EOD=0 / PP=0 indicates perfect fairness; '
    'red line = 0.10 threshold)',
    fontsize=11, y=1.03
)
plt.tight_layout()
fig_path = os.path.join(
    BASE_FIG, 'fig_fairness_metrics.png'
)
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.close()
print(f"\nFairness figure saved : {fig_path}")

fair_path = os.path.join(
    BASE_OUT, 'table_fairness_metrics.csv'
)
df_fair.to_csv(fair_path, index=False)
print(f"Fairness table saved  : {fair_path}")
print("\n=== 11_fairness_metrics.ipynb complete ===")

Configuration loaded.

  Hypertension

  Sensitive attribute: sex
    DPD (Demographic Parity Diff) : 0.2162
    EOD (Equal Opportunity Diff)  : 0.0802
    PP  (Predictive Parity Diff)  : 0.0549
    Positive rates by group:
      Female: 0.3611
      Male: 0.5773

  Sensitive attribute: age_group
    DPD (Demographic Parity Diff) : 0.6224
    EOD (Equal Opportunity Diff)  : 0.3558
    PP  (Predictive Parity Diff)  : 0.3039
    Positive rates by group:
      elderly: 0.8359
      middle: 0.4649
      young: 0.2135

  Diabetes

  Sensitive attribute: sex
    DPD (Demographic Parity Diff) : 0.1473
    EOD (Equal Opportunity Diff)  : 0.0135
    PP  (Predictive Parity Diff)  : 0.0298
    Positive rates by group:
      Female: 0.3341
      Male: 0.4814

  Sensitive attribute: age_group
    DPD (Demographic Parity Diff) : 0.7479
    EOD (Equal Opportunity Diff)  : 0.3522
    PP  (Predictive Parity Diff)  : 0.3768
    Positive rates by group:
      elderly: 0.8336
      middle: 0.4159
      yo